In [1]:
import pandas as pd
import sys
import os
sys.path.insert(0, '/home/hugo/gitlab.pavlovia/online_test/code')
from my_analysis.data_loader import load_and_aggregate
from my_analysis.utils import calculate_dice
from sklearn.metrics import cohen_kappa_score
import numpy as np
import datetime
import plotly.express as px

In [7]:
df = pd.read_csv('data/preload_data.csv')
print('They are ', df['judge_ID'].nunique(), 'participants')
df_analysis = df[df['done']].copy()
df_analysis = df_analysis.drop('Unnamed: 0', axis=1 )
df_analysis['clip_uid'] = df_analysis['scene_id'].astype(str) + "_" + df_analysis['player'].astype(str) + "_" + df_analysis['clip'].astype(str)
for scene in df_analysis['scene_id'].unique():
    df_f = df_analysis[df_analysis['scene_id']==scene]
    print('For scene ',scene,' they are', df_f['participant'].nunique(),'judge')
print(df_analysis['scene_id'].value_counts())
df_analysis.head()

They are  36 participants
For scene  w4l1s7  they are 35 judge
For scene  w3l1s1  they are 1 judge
scene_id
w4l1s7    10360
w3l1s1      400
Name: count, dtype: int64


,participant,judge_ID,scene_id,player,question,clip,answer,visualisation_time,clip_path,video_duration,done,clip_uid
0,692878,6914df8bbecd11a94b6ddb2f,w4l1s7,ppo,Q_exH,0,False,24.5030,sourcedata/ppo_mario_ep-20/sub-03/ses-003/beh/...,5.033333,True,w4l1s7_ppo_0
1,692878,6914df8bbecd11a94b6ddb2f,w4l1s7,ppo,Q_exH,1,False,19.8826,sourcedata/ppo_mario_ep-20/sub-03/ses-003/beh/...,4.816667,True,w4l1s7_ppo_1
2,692878,6914df8bbecd11a94b6ddb2f,w4l1s7,ppo,Q_exH,2,True,19.0279,sourcedata/ppo_mario_ep-20/sub-03/ses-003/beh/...,5.183333,True,w4l1s7_ppo_2
3,692878,6914df8bbecd11a94b6ddb2f,w4l1s7,ppo,Q_exH,3,False,20.1270,sourcedata/ppo_mario_ep-20/sub-03/ses-003/beh/...,2.450000,True,w4l1s7_ppo_3
4,692878,6914df8bbecd11a94b6ddb2f,w4l1s7,ppo,Q_exH,4,False,6.4104,sourcedata/ppo_mario_ep-20/sub-03/ses-003/beh/...,2.216667,True,w4l1s7_ppo_4


In [9]:
df_analysis = df_analysis[df_analysis['scene_id']=='w4l1s7']
df_analysis = df_analysis[df_analysis['judge_ID']!='66a9d46ff5d39bbede9f2e92']
print(df_analysis['judge_ID'].nunique())

34


In [10]:
# --- 3. Agreement Metrics Outliers (< Mean - 3*SD) ---
metric_outliers = set()
judge_metrics = {j: {'percent': [], 'kappa': [], 'dice': []} for j in df_analysis['judge_ID'].unique()}
    
for (scene, player), group in df_analysis.groupby(['scene_id', 'player']):
        # Pivot answers: Index=[question, clip], Columns=judge_ID
        pivoted = group.pivot_table(index=['question', 'clip'], columns='judge_ID', values='answer', aggfunc='first')
        for judge in pivoted.columns:
            others = pivoted.drop(columns=[judge])
            consensus = others.mode(axis=1).iloc[:, 0]
            judge_ans = pivoted[judge]
            valid_mask = judge_ans.notna() & consensus.notna()

            if not valid_mask.any(): continue
            
            v1 = judge_ans[valid_mask].astype(int)
            v2 = consensus[valid_mask].astype(int)
            
            judge_metrics[judge]['percent'].append((v1 == v2).mean())
            try:
                k = cohen_kappa_score(v1, v2)
                if np.isnan(k): k = 1.0 
                judge_metrics[judge]['kappa'].append(k)
            except: pass
            judge_metrics[judge]['dice'].append(calculate_dice(v1, v2))
    # Average metrics per judge
final_judge_metrics = []
for judge, measurements in judge_metrics.items():
        row = {'judge_ID': judge}
        for m in ['percent', 'kappa', 'dice']:
            row[m] = np.mean(measurements[m]) if measurements[m] else np.nan
        final_judge_metrics.append(row)           
df_metrics = pd.DataFrame(final_judge_metrics)
for metric in ['percent', 'kappa', 'dice']:
        mean_val = df_metrics[metric].mean()
        std_val = df_metrics[metric].std()
        threshold = mean_val - 1 * std_val
        print('Threshold for ',metric, ': ',threshold)
        outliers = df_metrics[df_metrics[metric] < threshold]['judge_ID'].tolist()
        metric_outliers.update(outliers)



Threshold for  percent :  0.582867992523312
Threshold for  kappa :  0.06473683881157746
Threshold for  dice :  0.33356767781566093


In [14]:
px.histogram(df_metrics, x='kappa')

In [13]:
px.scatter(df_metrics, x='kappa', y='dice', color='judge_ID')